In [2]:
#################################################
# 1. Load Credentials from .env
#################################################

import os

from dotenv import load_dotenv

import requests

from requests.auth import HTTPBasicAuth

import pandas as pd

import numpy as np

from pathlib import Path

load_dotenv()

USER = os.getenv("DBREPO_USER")

PASSWORD = os.getenv("DBREPO_PASSWORD")

print("Credentials loaded.")

Credentials loaded.


In [3]:
#################################################
# 2. DBRepo Client Setup
#################################################

from dbrepo.RestClient import RestClient

BASE_URL = "https://test.dbrepo.tuwien.ac.at"

client = RestClient(
    BASE_URL,
    username=USER,
    password=PASSWORD
)

print("DBRepo client initialized.")

DBRepo client initialized.


In [4]:
#################################################
# 3. Validate Database and Tables
#################################################

DB_NAME = "data_stew_grp22_air_quality"

databases = client.get_databases()

DB_ID = None

for db in databases:

    if db.name == DB_NAME:

        DB_ID = db.id

        print("Database found:")
        print("ID:", db.id)
        print("Name:", db.name)

        break

# Stop if DB does not exist
if DB_ID is None:

    raise Exception("Database not found.")

# Fetch tables
tables = client.get_tables(DB_ID)

print("\nExisting Tables:\n")

for table in tables:

    print("-", table.name)

# Validate required tables
required_tables = {
    "t_station",
    "t_time",
    "t_measurement"
}

existing_tables = {t.name for t in tables}

missing_tables = required_tables - existing_tables

if len(missing_tables) > 0:

    raise Exception(f"Missing tables: {missing_tables}")

print("\nAll required tables exist.")

Database found:
ID: 81d82941-cae0-4cba-b27d-1bd883dd713a
Name: data_stew_grp22_air_quality

Existing Tables:

- t_measurement
- t_time
- t_station

All required tables exist.


In [7]:
#################################################
# 4. Load Raw Source Dataset
#################################################

DATA_PATH = Path("../data/inp_air_quality_raw_v1.csv")

# Check file exists
if not DATA_PATH.exists():

    raise Exception(f"Dataset not found: {DATA_PATH}")

# Load raw dataset
df = pd.read_csv(DATA_PATH)

print("Raw dataset loaded successfully.\n")

print("Shape:", df.shape)

print("\nColumns:\n")

print(df.columns.tolist())

print("\nPreview:\n")

display(df.head())

Raw dataset loaded successfully.

Shape: (43800, 23)

Columns:

['Estación', 'Título', 'latitud', 'longitud', 'Fecha', 'Periodo', 'SO2', 'NO', 'NO2', 'CO', 'PM10', 'O3', 'dd', 'vv', 'TMP', 'HR', 'PRB', 'RS', 'LL', 'BEN', 'TOL', 'MXIL', 'PM25']

Preview:



,Estación,Título,latitud,longitud,Fecha,Periodo,SO2,NO,NO2,CO,PM10,O3,dd,vv,TMP,HR,PRB,RS,LL,BEN,TOL,MXIL,PM25
0,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,1,4.0,2.0,15.0,0.29,5.0,58.0,158.0,1.13,14.1,54.0,1009.0,17.0,0.0,0.10,0.60,0.40,3.0
1,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,2,3.0,1.0,14.0,0.25,4.0,54.0,203.0,0.48,14.1,55.0,1008.0,17.0,0.0,0.10,0.90,0.60,6.0
2,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,3,5.0,1.0,16.0,0.26,7.0,50.0,5.0,0.10,13.4,58.0,1007.0,17.0,0.0,0.10,0.92,1.88,6.0
3,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,4,4.0,1.0,5.0,0.24,6.0,68.0,181.0,0.75,15.3,53.0,1006.0,17.0,0.0,0.13,1.23,3.20,1.0
4,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,5,4.0,2.0,13.0,0.26,6.0,56.0,181.0,0.78,14.9,55.0,1006.0,17.0,0.0,0.10,0.50,0.63,1.0


In [9]:
#################################################
# 5. Inspect Raw Dataset Structure
#################################################

print("Dataset Information:\n")

print(df.info())

print("\nColumns:\n")

for col in df.columns:

    print("-", col)

print("\nMissing Values:\n")

print(df.isnull().sum())

print("\nPreview:\n")

display(df.head())

Dataset Information:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43800 entries, 0 to 43799
Data columns (total 23 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Estación  43800 non-null  int64  
 1   Título    43800 non-null  object 
 2   latitud   43800 non-null  float64
 3   longitud  43800 non-null  float64
 4   Fecha     43800 non-null  object 
 5   Periodo   43800 non-null  int64  
 6   SO2       43329 non-null  float64
 7   NO        43030 non-null  float64
 8   NO2       43078 non-null  float64
 9   CO        34563 non-null  float64
 10  PM10      34455 non-null  float64
 11  O3        43309 non-null  float64
 12  dd        17459 non-null  float64
 13  vv        17459 non-null  float64
 14  TMP       17459 non-null  float64
 15  HR        17459 non-null  float64
 16  PRB       17459 non-null  float64
 17  RS        17459 non-null  float64
 18  LL        17460 non-null  float64
 19  BEN       8430 non-null   float64
 20  TOL   

,Estación,Título,latitud,longitud,Fecha,Periodo,SO2,NO,NO2,CO,PM10,O3,dd,vv,TMP,HR,PRB,RS,LL,BEN,TOL,MXIL,PM25
0,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,1,4.0,2.0,15.0,0.29,5.0,58.0,158.0,1.13,14.1,54.0,1009.0,17.0,0.0,0.10,0.60,0.40,3.0
1,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,2,3.0,1.0,14.0,0.25,4.0,54.0,203.0,0.48,14.1,55.0,1008.0,17.0,0.0,0.10,0.90,0.60,6.0
2,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,3,5.0,1.0,16.0,0.26,7.0,50.0,5.0,0.10,13.4,58.0,1007.0,17.0,0.0,0.10,0.92,1.88,6.0
3,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,4,4.0,1.0,5.0,0.24,6.0,68.0,181.0,0.75,15.3,53.0,1006.0,17.0,0.0,0.13,1.23,3.20,1.0
4,1,Estación Avenida Constitución,43.529806,-5.673428,2013-12-31,5,4.0,2.0,13.0,0.26,6.0,56.0,181.0,0.78,14.9,55.0,1006.0,17.0,0.0,0.10,0.50,0.63,1.0


In [10]:
#################################################
# 6. Rename Raw Dataset Columns
#################################################

df = df.rename(columns={

    "Estación": "station_id",
    "Título": "station_name",

    "latitud": "latitude",
    "longitud": "longitude",

    "Fecha": "date",
    "Periodo": "hour",

    "dd": "wind_direction",
    "vv": "wind_speed",

    "TMP": "temperature",
    "HR": "humidity",
    "PRB": "pressure",
    "RS": "solar_radiation",
    "LL": "rain"
})

print("Columns renamed successfully.\n")

print(df.columns.tolist())

Columns renamed successfully.

['station_id', 'station_name', 'latitude', 'longitude', 'date', 'hour', 'SO2', 'NO', 'NO2', 'CO', 'PM10', 'O3', 'wind_direction', 'wind_speed', 'temperature', 'humidity', 'pressure', 'solar_radiation', 'rain', 'BEN', 'TOL', 'MXIL', 'PM25']


In [11]:
#################################################
# 7. Create Normalized Station Dimension Table
#################################################

df_station = (

    df[[
        "station_id",
        "station_name",
        "latitude",
        "longitude"
    ]]

    .drop_duplicates()

    .sort_values("station_id")

    .reset_index(drop=True)
)

print("Station dimension table created.\n")

print("Shape:", df_station.shape)

print("\nPreview:\n")

display(df_station.head())

Station dimension table created.

Shape: (5, 4)

Preview:



,station_id,station_name,latitude,longitude
0,1,Estación Avenida Constitución,43.529806,-5.673428
1,2,Estación Avenida Argentina,43.538869,-5.698930
2,3,Estación Avenida Hermanos Felgueroso,43.535061,-5.658123
3,4,Estación Avenida Castilla,43.537940,-5.645951
4,10,Estación de Montevil,43.517315,-5.672499


In [12]:
#################################################
# 8. Create Normalized Time Dimension Table
#################################################

# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Extract unique date-hour combinations
df_time = (

    df[[
        "date",
        "hour"
    ]]

    .drop_duplicates()

    .sort_values([
        "date",
        "hour"
    ])

    .reset_index(drop=True)
)

# Generate surrogate key
df_time["time_id"] = range(1, len(df_time) + 1)

# Reorder columns
df_time = df_time[[
    "time_id",
    "date",
    "hour"
]]

print("Time dimension table created.\n")

print("Shape:", df_time.shape)

print("\nPreview:\n")

display(df_time.head())

Time dimension table created.

Shape: (8760, 3)

Preview:



,time_id,date,hour
0,1,2013-01-01,1
1,2,2013-01-01,2
2,3,2013-01-01,3
3,4,2013-01-01,4
4,5,2013-01-01,5


In [15]:
#################################################
# 9. Create Normalized Measurement Fact Table
#################################################

# Create lookup table for time_id
df_time_lookup = df_time.copy()

# Merge time_id into original dataframe
df_measurement = df.merge(

    df_time_lookup,

    on=[
        "date",
        "hour"
    ],

    how="left"
)

# Generate surrogate measurement key
df_measurement["measurement_id"] = range(
    1,
    len(df_measurement) + 1
)

# Select final fact table columns
df_measurement = df_measurement[[

    "measurement_id",

    "station_id",
    "time_id",

    "SO2",
    "NO",
    "NO2",
    "CO",
    "PM10",
    "O3",
    "PM25",

    "wind_direction",
    "wind_speed",

    "temperature",
    "humidity",
    "pressure",
    "solar_radiation",
    "rain",

    "BEN",
    "TOL",
    "MXIL"
]]

# Reset index
df_measurement = df_measurement.reset_index(drop=True)

print("Measurement fact table created successfully.\n")

print("Shape:", df_measurement.shape)

print("\nPreview:\n")

display(df_measurement.head())

Measurement fact table created successfully.

Shape: (43800, 20)

Preview:



,measurement_id,station_id,time_id,SO2,NO,NO2,CO,PM10,O3,PM25,wind_direction,wind_speed,temperature,humidity,pressure,solar_radiation,rain,BEN,TOL,MXIL
0,1,1,8737,4.0,2.0,15.0,0.29,5.0,58.0,3.0,158.0,1.13,14.1,54.0,1009.0,17.0,0.0,0.10,0.60,0.40
1,2,1,8738,3.0,1.0,14.0,0.25,4.0,54.0,6.0,203.0,0.48,14.1,55.0,1008.0,17.0,0.0,0.10,0.90,0.60
2,3,1,8739,5.0,1.0,16.0,0.26,7.0,50.0,6.0,5.0,0.10,13.4,58.0,1007.0,17.0,0.0,0.10,0.92,1.88
3,4,1,8740,4.0,1.0,5.0,0.24,6.0,68.0,1.0,181.0,0.75,15.3,53.0,1006.0,17.0,0.0,0.13,1.23,3.20
4,5,1,8741,4.0,2.0,13.0,0.26,6.0,56.0,1.0,181.0,0.78,14.9,55.0,1006.0,17.0,0.0,0.10,0.50,0.63


In [16]:
#################################################
# 10. Validate Normalized Tables
#################################################

print("t_station Shape:")
print(df_station.shape)

print("\nt_time Shape:")
print(df_time.shape)

print("\nt_measurement Shape:")
print(df_measurement.shape)

print("\nMeasurement Table Columns:\n")

print(df_measurement.columns.tolist())

print("\nMissing Values in Measurement Table:\n")

print(df_measurement.isnull().sum())

print("\nValidation completed successfully.")

t_station Shape:
(5, 4)

t_time Shape:
(8760, 3)

t_measurement Shape:
(43800, 20)

Measurement Table Columns:

['measurement_id', 'station_id', 'time_id', 'SO2', 'NO', 'NO2', 'CO', 'PM10', 'O3', 'PM25', 'wind_direction', 'wind_speed', 'temperature', 'humidity', 'pressure', 'solar_radiation', 'rain', 'BEN', 'TOL', 'MXIL']

Missing Values in Measurement Table:

measurement_id         0
station_id             0
time_id                0
SO2                  471
NO                   770
NO2                  722
CO                  9237
PM10                9345
O3                   491
PM25               26344
wind_direction     26341
wind_speed         26341
temperature        26341
humidity           26341
pressure           26341
solar_radiation    26341
rain               26340
BEN                35370
TOL                35375
MXIL               35370
dtype: int64

Validation completed successfully.


In [19]:
#################################################
# 11. Inspect t_measurement Schema
#################################################

tables = client.get_tables(DB_ID)

measurement_table = None

for table in tables:

    if table.name == "t_measurement":

        measurement_table = client.get_table(DB_ID, table.id)

        break

if measurement_table is None:

    raise Exception("t_measurement not found.")

print("Table:", measurement_table.name)

print("\nColumns:\n")

for col in measurement_table.columns:

    print(
        f"{col.name} | Type: {col.type}"
    )

Table: t_measurement

Columns:

measurement_id | Type: ColumnType.BIGINT
station_id | Type: ColumnType.BIGINT
time_id | Type: ColumnType.BIGINT
SO2 | Type: ColumnType.DECIMAL
NO | Type: ColumnType.DECIMAL
NO2 | Type: ColumnType.DECIMAL
CO | Type: ColumnType.DECIMAL
PM10 | Type: ColumnType.DECIMAL
O3 | Type: ColumnType.DECIMAL
PM25 | Type: ColumnType.DECIMAL
wind_direction | Type: ColumnType.DECIMAL
wind_speed | Type: ColumnType.DECIMAL
temperature | Type: ColumnType.DECIMAL
humidity | Type: ColumnType.DECIMAL
pressure | Type: ColumnType.DECIMAL
solar_radiation | Type: ColumnType.DECIMAL
rain | Type: ColumnType.DECIMAL
BEN | Type: ColumnType.DECIMAL
TOL | Type: ColumnType.DECIMAL
MXIL | Type: ColumnType.DECIMAL


In [20]:
#################################################
# 12. Prepare Data for DBRepo Ingestion
#################################################

# Create ingestion copy
df_measurement_upload = df_measurement.copy()

# Fill missing numeric values
df_measurement_upload = df_measurement_upload.fillna(0)

print("Missing values after preprocessing:\n")

print(df_measurement_upload.isnull().sum())

print("\nData prepared successfully for DBRepo ingestion.")

Missing values after preprocessing:

measurement_id     0
station_id         0
time_id            0
SO2                0
NO                 0
NO2                0
CO                 0
PM10               0
O3                 0
PM25               0
wind_direction     0
wind_speed         0
temperature        0
humidity           0
pressure           0
solar_radiation    0
rain               0
BEN                0
TOL                0
MXIL               0
dtype: int64

Data prepared successfully for DBRepo ingestion.


In [21]:
#################################################
# 13. Check Existing Table Row Counts
#################################################

tables = client.get_tables(DB_ID)

for table in tables:

    table_data = client.get_table_data(
        DB_ID,
        table.id
    )

    print(f"{table.name}: {len(table_data)} rows")

t_measurement: 0 rows
t_time: 0 rows
t_station: 0 rows


In [37]:
#################################################
# 14. Working DBRepo Upload Patch
#################################################

import io

def patched_upload(self, dataframe):

    buffer = io.StringIO()

    dataframe.to_csv(
        buffer,
        index=False
    )

    url = '/api/v1/upload'

    response = self._wrapper(

        method="post",

        url=url,

        force_auth=True,

        files={
            'file': (
                'dataframe.csv',
                buffer.getvalue()
            )
        }
    )

    print("Upload status:", response.status_code)

    # Correct behavior for TU Wien DBRepo server
    if response.status_code == 201:

        s3_key = response.headers.get("X-S3-Key")

        if s3_key is None:

            raise Exception(
                "Upload succeeded but X-S3-Key header missing."
            )

        print("S3 Key:", s3_key)

        return s3_key

    raise Exception(
        f"Upload failed: {response.status_code}"
    )

# Apply monkey patch
client._upload = patched_upload.__get__(client)

print("Final DBRepo upload patch applied.")

Final DBRepo upload patch applied.


In [39]:
#################################################
# 15. Validate t_station Import
#################################################

import time

# Wait for async import
time.sleep(5)

# Check row count
station_count = client.get_table_data_count(

    database_id=DB_ID,

    table_id=station_table_id
)

print("Rows in t_station:", station_count)

# Preview imported data
df_station_db = client.get_table_data(

    database_id=DB_ID,

    table_id=station_table_id,

    page=0,

    size=10
)

display(df_station_db)

Rows in t_station: 5


,station_id,station_name,latitude,longitude
0,1,Estación Avenida Constitución,43.529806,-5.673428
1,2,Estación Avenida Argentina,43.538869,-5.698930
2,3,Estación Avenida Hermanos Felgueroso,43.535061,-5.658123
3,4,Estación Avenida Castilla,43.537940,-5.645951
4,10,Estación de Montevil,43.517315,-5.672499


In [40]:
#################################################
# 16. Upload t_time
#################################################

# Get table ID
tables = client.get_tables(DB_ID)

time_table_id = None

for table in tables:

    if table.name == "t_time":

        time_table_id = table.id

        break

if time_table_id is None:

    raise Exception("t_time not found.")

# Upload dataframe
client.import_table_data(

    database_id=DB_ID,

    table_id=time_table_id,

    dataframe=df_time
)

print("t_time upload initiated.")

Upload status: 201
S3 Key: dataframe.csv
t_time upload initiated.


In [41]:
#################################################
# 17. Validate t_time Import
#################################################

import time

# Wait for async import
time.sleep(5)

# Check row count
time_count = client.get_table_data_count(

    database_id=DB_ID,

    table_id=time_table_id
)

print("Rows in t_time:", time_count)

# Preview imported data
df_time_db = client.get_table_data(

    database_id=DB_ID,

    table_id=time_table_id,

    page=0,

    size=10
)

display(df_time_db)

Rows in t_time: 8760


,time_id,date,hour
0,1,2013-01-01,1
1,2,2013-01-01,2
2,3,2013-01-01,3
3,4,2013-01-01,4
4,5,2013-01-01,5
5,6,2013-01-01,6
6,7,2013-01-01,7
7,8,2013-01-01,8
8,9,2013-01-01,9
9,10,2013-01-01,10


In [44]:
#################################################
# 19. Create Clean Measurement Data
#################################################

df_measurement_clean = df_measurement.copy()

# Fill missing numeric values
numeric_columns = [

    "SO2",
    "NO",
    "NO2",
    "CO",
    "PM10",
    "O3",
    "PM25",
    "wind_direction",
    "wind_speed",
    "temperature",
    "humidity",
    "pressure",
    "solar_radiation",
    "rain",
    "BEN",
    "TOL",
    "MXIL"
]

for col in numeric_columns:

    df_measurement_clean[col] = (
        df_measurement_clean[col]
        .fillna(
            df_measurement_clean[col].median()
        )
    )

print("Missing values after preprocessing:\n")

print(df_measurement_clean.isnull().sum())

print("\nClean measurement dataframe ready.")

Missing values after preprocessing:

measurement_id     0
station_id         0
time_id            0
SO2                0
NO                 0
NO2                0
CO                 0
PM10               0
O3                 0
PM25               0
wind_direction     0
wind_speed         0
temperature        0
humidity           0
pressure           0
solar_radiation    0
rain               0
BEN                0
TOL                0
MXIL               0
dtype: int64

Clean measurement dataframe ready.


In [45]:
#################################################
# 20. Upload Clean t_measurement
#################################################

print("Rows to upload:")

print(len(df_measurement_clean))

# Upload cleaned dataframe
client.import_table_data(

    database_id=DB_ID,

    table_id=measurement_table_id,

    dataframe=df_measurement_clean
)

print("\nt_measurement upload initiated.")

Rows to upload:
43800
Upload status: 201
S3 Key: dataframe.csv

t_measurement upload initiated.


In [46]:
#################################################
# 21. Validate t_measurement Import
#################################################

import time

# Wait for async import
time.sleep(10)

# Check row count
measurement_count = client.get_table_data_count(

    database_id=DB_ID,

    table_id=measurement_table_id
)

print("Rows in t_measurement:")

print(measurement_count)

# Preview imported data
df_measurement_db = client.get_table_data(

    database_id=DB_ID,

    table_id=measurement_table_id,

    page=0,

    size=10
)

display(df_measurement_db)

Rows in t_measurement:
43800


,measurement_id,station_id,time_id,so2,no,no2,co,pm10,o3,pm25,wind_direction,wind_speed,temperature,humidity,pressure,solar_radiation,rain,ben,tol,mxil
0,1,1,8737,4.0,2.0,15.0,0.29,5.0,58.0,3.0,158.0,1.13,14.1,54.0,1009.0,17.0,0.0,0.10,0.60,0.40
1,2,1,8738,3.0,1.0,14.0,0.25,4.0,54.0,6.0,203.0,0.48,14.1,55.0,1008.0,17.0,0.0,0.10,0.90,0.60
2,3,1,8739,5.0,1.0,16.0,0.26,7.0,50.0,6.0,5.0,0.10,13.4,58.0,1007.0,17.0,0.0,0.10,0.92,1.88
3,4,1,8740,4.0,1.0,5.0,0.24,6.0,68.0,1.0,181.0,0.75,15.3,53.0,1006.0,17.0,0.0,0.13,1.23,3.20
4,5,1,8741,4.0,2.0,13.0,0.26,6.0,56.0,1.0,181.0,0.78,14.9,55.0,1006.0,17.0,0.0,0.10,0.50,0.63
5,6,1,8742,4.0,4.0,23.0,0.30,8.0,42.0,5.0,180.0,0.32,14.3,57.0,1006.0,17.0,0.0,0.10,0.73,0.60
6,7,1,8743,7.0,33.0,51.0,0.42,9.0,11.0,3.0,318.0,0.10,13.1,60.0,1006.0,17.0,0.0,0.15,1.20,1.27
7,8,1,8744,15.0,52.0,63.0,0.63,19.0,4.0,3.0,337.0,0.10,12.4,61.0,1006.0,17.0,0.0,0.23,1.85,1.70
8,9,1,8745,9.0,87.0,60.0,0.64,18.0,16.0,5.0,180.0,1.05,12.7,58.0,1005.0,17.0,0.0,0.53,3.08,2.45
9,10,1,8746,8.0,67.0,71.0,0.61,22.0,10.0,10.0,316.0,0.12,14.1,53.0,1006.0,28.0,0.0,0.52,3.15,2.72
